# 📺 Lab 01 — Previously, everything worked
A fictional streaming service bills three purchases in USD. Create both versions of the synthetic feed and pin the contract before watching a failure. One row means one purchase. Money is in cents and `tv_app` is the canonical channel.

Run in a new, disposable `zeroops_lab_<alias>` schema. This notebook replaces only the two named fixture tables. Bundle tasks supply `catalog` and `schema`. For an interactive run, set the named text parameters in the notebook first.


In [ ]:
%sql
SELECT assert_true(:catalog RLIKE '^[A-Za-z_][A-Za-z0-9_]*$' AND :schema RLIKE '^zeroops_lab_[a-z0-9_]+$', 'Use a dedicated zeroops_lab_<alias> schema') AS scope_check;


In [ ]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.' || :schema || '.zops_feed_v1')
USING DELTA COMMENT 'Synthetic purchases. Grain: purchase_id. USD values in cents.' AS
SELECT * FROM VALUES
  (1, 'tv_app', 'USD', 1200),
  (2, 'tv_app', 'USD', 800),
  (3, 'web', 'USD', 500)
AS purchases(purchase_id, channel, currency, amount_cents);


In [ ]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.' || :schema || '.zops_feed_v2')
USING DELTA COMMENT 'Synthetic v1 replay. amount_minor is USD cents under its new field name.' AS
SELECT purchase_id, channel, currency, amount_cents AS amount_minor
FROM IDENTIFIER(:catalog || '.' || :schema || '.zops_feed_v1');


## The control
Expect three unique purchases, one currency (USD), 2,500 cents total and two TV-app purchases worth 2,000 cents. The assertion returns `NULL` on agreement and raises on a mismatch. Run Lab 02 with the v1 input before switching to v2.


In [ ]:
%sql
SELECT COUNT(*) AS purchases, COUNT(DISTINCT purchase_id) AS distinct_purchases,
       SUM(amount_cents) AS total_cents, COUNT_IF(channel = 'tv_app') AS tv_purchases,
       SUM(CASE WHEN channel = 'tv_app' THEN amount_cents ELSE 0 END) AS tv_cents,
       assert_true(COUNT(*) = 3 AND COUNT(DISTINCT purchase_id) = 3
         AND COUNT_IF(currency = 'USD') = 3 AND SUM(amount_cents) = 2500
         AND COUNT_IF(channel = 'tv_app') = 2
         AND SUM(CASE WHEN channel = 'tv_app' THEN amount_cents ELSE 0 END) = 2000,
         'Synthetic v1 fixture changed') AS contract_check
FROM IDENTIFIER(:catalog || '.' || :schema || '.zops_feed_v1');
